## Этап 1. Протокол запуска

Этот notebook воспроизводит этап 3 RuSearchRank для модели cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 в ревизии 1427fd652930e4ba29e8149678df786c240d8825. Cross-encoder — модель, которая совместно обрабатывает запрос и документ и оценивает их релевантность.

Подключите закрытый набор данных Kaggle с официальными ZIP этапов 1 и 2, включите один GPU и выполните Restart Kernel and Run All. Ручное продолжение не требуется. До выбора контрольной точки dev qrels не загружаются и не разбираются. Полное обучение выполняется последовательно: C1 → A1 → A2 → B1.


## Этап 2. Константы и неизменяемый выпуск

Все пути выводятся из корня Kaggle. Тег выпуска и ревизия модели считаются частью технического контракта; плавающая ветка не используется.


In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

KAGGLE_ROOT = Path("/kaggle")
KAGGLE_WORKING = KAGGLE_ROOT / "working"
KAGGLE_INPUT = KAGGLE_ROOT / "input"
REPOSITORY = KAGGLE_WORKING / "ru-search-rank"
REPOSITORY_URL = "https://github.com/kopanevk/ru-search-rank.git"
RELEASE_REF = "phase3-v1.0.1"
MODEL_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
MODEL_REVISION = "1427fd652930e4ba29e8149678df786c240d8825"
TRAIN_QRELS_SHA256 = "bf1f737cda0d66bc38fef5f9d91843f7a89428c5c5a8a3dce4764f527ec344ef"
ALLOW_OVERWRITE_PHASE3 = False
RESUME_TRAINING = True

if ALLOW_OVERWRITE_PHASE3 and RESUME_TRAINING:
    raise RuntimeError("--resume и --overwrite нельзя включать одновременно")

def run_checked(command, *, cwd=None, env=None, stream=False):
    actual_cwd = Path(cwd) if cwd is not None else (
        REPOSITORY if REPOSITORY.is_dir() else KAGGLE_WORKING
    )
    print("$", " ".join(map(str, command)), flush=True)
    result = subprocess.run(
        list(map(str, command)),
        cwd=actual_cwd,
        env=env,
        text=True,
        capture_output=not stream,
    )
    if not stream:
        if result.stdout:
            print(result.stdout.rstrip())
        if result.stderr:
            print(result.stderr.rstrip(), file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Команда завершилась с кодом {result.returncode}; "
            "полный журнал сохранён выше."
        )
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def phase3_options():
    return ["--overwrite"] if ALLOW_OVERWRITE_PHASE3 else []

def training_options(run_id):
    if globals().get("FINETUNE_SMOKE_PASSED") is not True:
        raise RuntimeError("До обучения должна пройти полная первичная проверка")
    if ALLOW_OVERWRITE_PHASE3:
        return ["--overwrite"]
    manifest = REPOSITORY / f"artifacts/models/{run_id}/run_manifest.json"
    return ["--resume"] if RESUME_TRAINING and manifest.is_file() else []


## Этап 3. Первичная проверка GPU, RAM и диска

Проверка завершается до скачивания модели. Объявленные пороги памяти и диска используются как жёсткие условия.


In [ ]:
if platform.system() != "Linux":
    raise RuntimeError("Полный протокол этапа 3 требует Linux")
for required_path in (KAGGLE_WORKING, KAGGLE_INPUT):
    if not required_path.is_dir():
        raise RuntimeError(f"Не найден обязательный каталог Kaggle: {required_path}")

gpu_probe = run_checked(["nvidia-smi"], cwd=KAGGLE_WORKING)
gpu_list = run_checked(["nvidia-smi", "-L"], cwd=KAGGLE_WORKING).stdout.strip()
if not gpu_list:
    raise RuntimeError("Kaggle не предоставил GPU")

meminfo = {}
for line in Path("/proc/meminfo").read_text(encoding="utf-8").splitlines():
    key, value = line.split(":", 1)
    meminfo[key] = int(value.strip().split()[0]) * 1024
disk = shutil.disk_usage(KAGGLE_WORKING)
MIN_TOTAL_RAM_GIB = 12
MIN_AVAILABLE_RAM_GIB = 4
MIN_FREE_DISK_GIB = 25
if meminfo["MemTotal"] < MIN_TOTAL_RAM_GIB * 1024**3:
    raise RuntimeError("Требуется не менее 12 ГиБ общей оперативной памяти")
if meminfo["MemAvailable"] < MIN_AVAILABLE_RAM_GIB * 1024**3:
    raise RuntimeError("Требуется не менее 4 ГиБ доступной оперативной памяти")
if disk.free < MIN_FREE_DISK_GIB * 1024**3:
    raise RuntimeError("Требуется не менее 25 ГиБ свободного места")
print(
    {
        "gpu": gpu_list.splitlines(),
        "ram_total_gib": round(meminfo["MemTotal"] / 1024**3, 2),
        "ram_available_gib": round(meminfo["MemAvailable"] / 1024**3, 2),
        "disk_free_gib": round(disk.free / 1024**3, 2),
    }
)


## Этап 4. Получение точной версии репозитория

Репозиторий клонируется по тегу выпуска. Повторный запуск принимает только ту же чистую фиксацию Git; обновление текущей ветки не выполняется.


In [ ]:
if not REPOSITORY.exists():
    run_checked(
        [
            "git",
            "clone",
            "--branch",
            RELEASE_REF,
            "--single-branch",
            REPOSITORY_URL,
            REPOSITORY,
        ],
        cwd=KAGGLE_WORKING,
        stream=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Каталог не является Git-репозиторием: {REPOSITORY}")

release_commit = run_checked(["git", "rev-parse", "HEAD"]).stdout.strip()
exact_tag = run_checked(
    ["git", "describe", "--tags", "--exact-match", "HEAD"]
).stdout.strip()
if exact_tag != RELEASE_REF:
    raise RuntimeError(
        f"HEAD не соответствует неизменяемому тегу {RELEASE_REF}: {exact_tag}"
    )
if run_checked(["git", "status", "--short"]).stdout.strip():
    raise RuntimeError("Рабочее дерево выпуска должно быть чистым")

commit_record = KAGGLE_WORKING / "rusearchrank-release-commit.txt"
if commit_record.is_file():
    if commit_record.read_text(encoding="utf-8").strip() != release_commit:
        raise RuntimeError("Тег выпуска указывает на другую фиксацию Git, чем при первом запуске")
else:
    commit_record.write_text(release_commit + "\n", encoding="utf-8")
print({"release_ref": RELEASE_REF, "release_commit": release_commit})


## Этап 5. Сборка и проверка trec_eval

Официальный trec_eval v9.0.8 собирается неизменённой командой make. Исполняемый файл размещается в рабочем каталоге, а код проекта получает его через TREC_EVAL_PATH.


In [ ]:
import re

TREC_EVAL_TAG = "v9.0.8"
TREC_EVAL_COMMIT = "d95ca64e14a47d763ae349fb65e6d8cde4141dbd"
TREC_SOURCE = KAGGLE_WORKING / "trec-eval-v9.0.8"
TREC_BINARY = KAGGLE_WORKING / "bin" / "trec_eval"
TREC_PROVENANCE = (
    REPOSITORY / "artifacts/work/phase2/trec_eval_build_provenance.json"
)

for tool in ("git", "make", "cc"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"Не найден инструмент сборки: {tool}")

if TREC_BINARY.is_file() and TREC_PROVENANCE.is_file():
    provenance = json.loads(TREC_PROVENANCE.read_text(encoding="utf-8"))
else:
    if TREC_SOURCE.exists() or TREC_BINARY.exists() or TREC_PROVENANCE.exists():
        raise RuntimeError(
            "Обнаружена неполная предыдущая сборка trec_eval; "
            "сохраните диагностику и начните с чистой рабочей сессии."
        )
    run_checked(
        [
            "git",
            "clone",
            "--branch",
            TREC_EVAL_TAG,
            "--single-branch",
            "https://github.com/usnistgov/trec_eval.git",
            TREC_SOURCE,
        ],
        cwd=KAGGLE_WORKING,
        stream=True,
    )
    head = run_checked(["git", "rev-parse", "HEAD"], cwd=TREC_SOURCE).stdout.strip()
    tags = run_checked(
        ["git", "tag", "--points-at", "HEAD"], cwd=TREC_SOURCE
    ).stdout.splitlines()
    if head != TREC_EVAL_COMMIT or TREC_EVAL_TAG not in tags:
        raise RuntimeError(f"Получена неожиданная ревизия trec_eval: {head}")
    makefile_sha256 = sha256_file(TREC_SOURCE / "Makefile")
    jobs = os.cpu_count() or 2
    build_command = f"make -j{jobs}"
    run_checked(["make", f"-j{jobs}"], cwd=TREC_SOURCE, stream=True)
    run_checked(["git", "diff", "--quiet", "HEAD"], cwd=TREC_SOURCE)
    run_checked(["git", "diff", "--cached", "--quiet"], cwd=TREC_SOURCE)
    if sha256_file(TREC_SOURCE / "Makefile") != makefile_sha256:
        raise RuntimeError("Makefile trec_eval изменился во время сборки")
    TREC_BINARY.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(TREC_SOURCE / "trec_eval", TREC_BINARY)
    TREC_BINARY.chmod(0o755)
    version_result = run_checked([TREC_BINARY, "-v"], cwd=KAGGLE_WORKING)
    version_text = (version_result.stdout or "") + (version_result.stderr or "")
    if re.search(r"\b9\.0\.7\b", version_text) is None:
        raise RuntimeError("Официальная сборка v9.0.8 должна сообщать версию 9.0.7")
    compiler = run_checked(["cc", "--version"], cwd=KAGGLE_WORKING).stdout.splitlines()[0]
    provenance = {
        "source_repository": "https://github.com/usnistgov/trec_eval.git",
        "source_tag": TREC_EVAL_TAG,
        "source_commit": TREC_EVAL_COMMIT,
        "source_tree_clean": True,
        "fresh_checkout": True,
        "source_path": str(TREC_SOURCE.resolve()),
        "makefile_sha256": makefile_sha256,
        "binary_path": str(TREC_BINARY.resolve()),
        "binary_sha256": sha256_file(TREC_BINARY),
        "binary_reported_version": "9.0.7",
        "expected_release_version": "9.0.8",
        "known_upstream_version_string_mismatch": True,
        "build_command": build_command,
        "compiler": compiler,
        "built_at": datetime.now(timezone.utc).isoformat(),
    }
    TREC_PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
    TREC_PROVENANCE.write_text(
        json.dumps(provenance, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

if provenance.get("source_commit") != TREC_EVAL_COMMIT:
    raise RuntimeError("Сведения о происхождении trec_eval содержат другую ревизию")
if provenance.get("binary_sha256") != sha256_file(TREC_BINARY):
    raise RuntimeError("Контрольная сумма trec_eval не совпала")
if not TREC_BINARY.is_file() or not os.access(TREC_BINARY, os.X_OK):
    raise RuntimeError("trec_eval не является исполняемым обычным файлом")
version_result = run_checked([TREC_BINARY, "-v"], cwd=KAGGLE_WORKING)
version_text = (version_result.stdout or "") + (version_result.stderr or "")
if re.search(r"\b9\.0\.7\b", version_text) is None:
    raise RuntimeError("Проверка версии trec_eval не пройдена")
os.environ["TREC_EVAL_PATH"] = str(TREC_BINARY.resolve())
print(
    {
        "trec_eval": os.environ["TREC_EVAL_PATH"],
        "binary_sha256": provenance["binary_sha256"],
        "source_commit": provenance["source_commit"],
    }
)


## Этап 6. Зафиксированное окружение

Используется Python 3.12.13 и полностью зафиксированный requirements/kaggle.lock. Бесконтрольное обновление зависимостей запрещено; проект устанавливается из уже проверенного тега без зависимостей.


In [ ]:
import venv

EXPECTED_PYTHON = "3.12.13"
if platform.python_version() != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Требуется Python {EXPECTED_PYTHON}, получен {platform.python_version()}. "
        "Этот выпуск не заявляет побайтовую воспроизводимость в другом Python."
    )

VENV_PATH = KAGGLE_WORKING / "rusearchrank-phase3-venv"
RUN_PYTHON_PATH = VENV_PATH / "bin" / "python"
if not RUN_PYTHON_PATH.is_file():
    venv.EnvBuilder(
        with_pip=True,
        system_site_packages=True,
        clear=False,
    ).create(VENV_PATH)
RUN_PYTHON = str(RUN_PYTHON_PATH)
python_probe = run_checked(
    [
        RUN_PYTHON,
        "-c",
        "import platform; print(platform.python_version())",
    ],
    cwd=REPOSITORY,
).stdout.strip()
if python_probe != EXPECTED_PYTHON:
    raise RuntimeError(f"Виртуальное окружение использует Python {python_probe}")

LOCK_PATH = REPOSITORY / "requirements/kaggle.lock"
if not LOCK_PATH.is_file():
    raise RuntimeError("Не найден requirements/kaggle.lock")
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-deps",
        "--requirement",
        LOCK_PATH,
    ],
    cwd=REPOSITORY,
    stream=True,
)
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-deps",
        "--editable",
        ".",
    ],
    cwd=REPOSITORY,
    stream=True,
)
version_probe = run_checked(
    [
        RUN_PYTHON,
        "-c",
        (
            "import json, platform, torch, transformers, tokenizers; "
            "print(json.dumps({'python': platform.python_version(), "
            "'torch': torch.__version__, 'transformers': transformers.__version__, "
            "'tokenizers': tokenizers.__version__, 'cuda': torch.version.cuda, "
            "'cuda_available': torch.cuda.is_available()}))"
        ),
    ],
    cwd=REPOSITORY,
)
versions = json.loads(version_probe.stdout)
expected_versions = {
    "python": "3.12.13",
    "torch": "2.13.0",
    "transformers": "5.14.1",
    "tokenizers": "0.22.2",
}
for name, expected in expected_versions.items():
    if versions.get(name) != expected:
        raise RuntimeError(
            f"Версия {name} не совпала: {versions.get(name)} != {expected}"
        )
if versions.get("cuda_available") is not True:
    raise RuntimeError("PyTorch не видит CUDA в зафиксированном окружении")
print(versions)


## Этап 7. Проверка ревизии модели и токенизатора

Ревизии сверяются с конфигурацией и эталонными контрольными суммами. Токенизатор сохраняется в отдельном неизменяемом каталоге и затем открывается без сети.


In [ ]:
config_json = run_checked(
    [
        RUN_PYTHON,
        "-c",
        (
            "import json, yaml; from pathlib import Path; "
            "print(json.dumps(yaml.safe_load("
            "Path('configs/finetune.yaml').read_text(encoding='utf-8'))))"
        ),
    ],
    cwd=REPOSITORY,
).stdout
config = json.loads(config_json)
golden = json.loads(
    (REPOSITORY / "tests/fixtures/pair_encoding_golden.json").read_text(
        encoding="utf-8"
    )
)
for label, actual, expected in (
    ("model id", config["base_model"]["id"], MODEL_ID),
    ("model revision", config["base_model"]["revision"], MODEL_REVISION),
    ("golden model revision", golden["model_revision"], MODEL_REVISION),
    ("tokenizer revision", config["base_model"]["tokenizer_revision"], MODEL_REVISION),
):
    if actual != expected:
        raise RuntimeError(f"Не совпал {label}: {actual}")

PINNED_TOKENIZER_DIR = KAGGLE_WORKING / "rusearchrank-pinned-tokenizer"
PINNED_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
download_script = r'''
import hashlib
import json
import shutil
import sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

repo_id, revision, destination, expected_json = sys.argv[1:]
destination = Path(destination)
expected = json.loads(expected_json)
info = HfApi().model_info(repo_id=repo_id, revision=revision)
if info.sha != revision:
    raise RuntimeError(f"model revision mismatch: {info.sha} != {revision}")
for name, digest in expected.items():
    target = destination / name
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != digest:
        cached = Path(
            hf_hub_download(repo_id=repo_id, filename=name, revision=revision)
        )
        temporary = target.with_name(target.name + ".tmp")
        shutil.copyfile(cached, temporary)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"tokenizer SHA-256 mismatch: {name}")
        temporary.replace(target)
'''
run_checked(
    [
        RUN_PYTHON,
        "-c",
        download_script,
        MODEL_ID,
        MODEL_REVISION,
        PINNED_TOKENIZER_DIR,
        json.dumps(golden["tokenizer_payload_sha256"], sort_keys=True),
    ],
    cwd=REPOSITORY,
    stream=True,
)
for name, expected in golden["tokenizer_payload_sha256"].items():
    actual = sha256_file(PINNED_TOKENIZER_DIR / name)
    if actual != expected:
        raise RuntimeError(f"Не совпала контрольная сумма токенизатора: {name}")
os.environ["RUSEARCHRANK_PINNED_TOKENIZER_DIR"] = str(
    PINNED_TOKENIZER_DIR.resolve()
)
run_checked(
    [
        RUN_PYTHON,
        "-c",
        (
            "import os; from transformers import AutoTokenizer; "
            "p=os.environ['RUSEARCHRANK_PINNED_TOKENIZER_DIR']; "
            "t=AutoTokenizer.from_pretrained(p, local_files_only=True); "
            "print({'tokenizer_class': type(t).__name__, 'vocabulary_size': len(t)})"
        ),
    ],
    cwd=REPOSITORY,
)


## Этап 8. Локальные тесты

Основной набор тестов выполняется с доступным GPU, но без теста, который намеренно требует отсутствия CUDA. Этот отрицательный сценарий запускается отдельно в дочернем процессе с CUDA_VISIBLE_DEVICES, равным пустой строке.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "pytest",
        "-q",
        "-k",
        "not test_unknown_run_and_non_cuda_production_are_blocked",
    ],
    cwd=REPOSITORY,
    stream=True,
)
cuda_negative_environment = os.environ.copy()
cuda_negative_environment["CUDA_VISIBLE_DEVICES"] = ""
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "pytest",
        "-q",
        "tests/test_training_loop.py::test_unknown_run_and_non_cuda_production_are_blocked",
    ],
    cwd=REPOSITORY,
    env=cuda_negative_environment,
    stream=True,
)
run_checked(
    [RUN_PYTHON, "scripts/validate_phase3_notebook.py"],
    cwd=REPOSITORY,
    stream=True,
)
run_checked(
    [RUN_PYTHON, "-m", "compileall", "-q", "src"],
    cwd=REPOSITORY,
)


## Этап 9. Восстановление этапов 1 и 2

Каждый входной ZIP проверяется по CRC и SHA-256. Файлы извлекаются во временный каталог; существующий файл репозитория никогда не заменяется другими байтами.


In [ ]:
PHASE_MARKERS = {
    "phase1": "candidate_cache_manifest.json",
    "phase2": "rerank_manifest.json",
}
archive_matches = {phase: [] for phase in PHASE_MARKERS}
for candidate in sorted(KAGGLE_INPUT.rglob("*.zip")):
    try:
        with zipfile.ZipFile(candidate) as archive:
            names = archive.namelist()
            if archive.testzip() is not None:
                raise RuntimeError(f"CRC-проверка не пройдена: {candidate}")
            basenames = {PurePosixPath(name).name for name in names}
    except zipfile.BadZipFile as exc:
        raise RuntimeError(f"Повреждён ZIP: {candidate}") from exc
    phases = [
        phase for phase, marker in PHASE_MARKERS.items() if marker in basenames
    ]
    if len(phases) > 1:
        raise RuntimeError(f"ZIP содержит манифесты двух этапов: {candidate}")
    if phases:
        archive_matches[phases[0]].append(candidate)

for phase, matches in archive_matches.items():
    if len(matches) != 1:
        raise RuntimeError(
            f"Для {phase} требуется ровно один ZIP, найдено {len(matches)}"
        )

phase_archive_sha256 = {}
for phase in ("phase1", "phase2"):
    source_zip = archive_matches[phase][0]
    phase_archive_sha256[phase] = sha256_file(source_zip)
    with tempfile.TemporaryDirectory(
        prefix=f"{phase}-restore-", dir=KAGGLE_WORKING
    ) as temporary_directory:
        staging = Path(temporary_directory)
        with zipfile.ZipFile(source_zip) as archive:
            names = archive.namelist()
            for name in names:
                member = PurePosixPath(name)
                if member.is_absolute() or ".." in member.parts:
                    raise RuntimeError(f"Небезопасный путь в ZIP: {name}")
                target = staging / member.as_posix()
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(name) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
        marker_matches = list(staging.rglob(PHASE_MARKERS[phase]))
        if len(marker_matches) != 1:
            raise RuntimeError(f"Не найден единственный манифест {phase}")
        for source in sorted(path for path in staging.rglob("*") if path.is_file()):
            relative = source.relative_to(staging)
            destination = REPOSITORY / relative
            if destination.exists():
                if not destination.is_file() or sha256_file(destination) != sha256_file(source):
                    raise RuntimeError(
                        f"Запрещена замена артефакта этапа 1/2: {relative}"
                    )
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, destination)
            if sha256_file(destination) != sha256_file(source):
                raise RuntimeError(f"Ошибка восстановления: {relative}")

archive_hash_path = (
    REPOSITORY / "artifacts/work/phase3/phase12_archive_sha256.json"
)
archive_hash_path.parent.mkdir(parents=True, exist_ok=True)
archive_hash_path.write_text(
    json.dumps(phase_archive_sha256, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(phase_archive_sha256)


## Этап 10. Получение train qrels

Сначала используется локальный файл с точной контрольной суммой. Если его нет, выполняется скачивание по зафиксированному URL. Dev qrels на этой стадии не загружается.


In [ ]:
CONFIG = "configs/finetune.yaml"
train_qrels_relative = config["inputs"]["train_qrels"]
train_qrels_path = REPOSITORY / train_qrels_relative

def valid_train_qrels(path):
    return (
        Path(path).is_file()
        and sha256_file(path) == TRAIN_QRELS_SHA256
    )

if not valid_train_qrels(train_qrels_path):
    local_candidates = [
        path
        for path in sorted(KAGGLE_INPUT.rglob(train_qrels_path.name))
        if valid_train_qrels(path)
    ]
    if len(local_candidates) > 1:
        raise RuntimeError("Найдено несколько локальных train qrels с нужной суммой")
    train_qrels_path.parent.mkdir(parents=True, exist_ok=True)
    if local_candidates:
        shutil.copyfile(local_candidates[0], train_qrels_path)
    else:
        import urllib.request

        train_url = run_checked(
            [
                RUN_PYTHON,
                "-c",
                (
                    "import yaml; from pathlib import Path; "
                    "c=yaml.safe_load(Path('configs/retrieval.yaml').read_text()); "
                    "print(c['dataset']['qrels']['train'])"
                ),
            ],
            cwd=REPOSITORY,
        ).stdout.strip()
        temporary_qrels = train_qrels_path.with_name(
            train_qrels_path.name + ".tmp"
        )
        with urllib.request.urlopen(train_url, timeout=120) as response:
            payload = response.read()
        temporary_qrels.write_bytes(payload)
        if not valid_train_qrels(temporary_qrels):
            actual = sha256_file(temporary_qrels)
            raise RuntimeError(
                f"SHA-256 train qrels не совпал: {actual}"
            )
        temporary_qrels.replace(train_qrels_path)

if not valid_train_qrels(train_qrels_path):
    raise RuntimeError("Итоговая проверка train qrels не пройдена")
print(
    {
        "path": train_qrels_relative,
        "sha256": sha256_file(train_qrels_path),
        "size_bytes": train_qrels_path.stat().st_size,
    }
)


## Этап 11. Построение разделения и обучающих пар

Разделение запросов и три режима пар строятся только по обучающим данным. Повторный запуск переиспользует идентичные артефакты.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "build-training-split",
        "--config",
        CONFIG,
    ]
    + phase3_options(),
    cwd=REPOSITORY,
    stream=True,
)
for regime in ("judged_only", "weak_negatives", "control_c1"):
    run_checked(
        [
            RUN_PYTHON,
            "-m",
            "rusearchrank.cli",
            "build-training-pairs",
            "--config",
            CONFIG,
            "--regime",
            regime,
        ]
        + phase3_options(),
        cwd=REPOSITORY,
        stream=True,
    )
pairs_manifest = json.loads(
    (REPOSITORY / "reports/audit/pairs_manifest.json").read_text(
        encoding="utf-8"
    )
)
print(
    {
        regime: {
            "usable_query_count": section["usable_query_count"],
            "pair_count": section["pair_count"],
            "pair_file_sha256": section["pair_file_sha256"],
        }
        for regime, section in pairs_manifest["regimes"].items()
    }
)


## Этап 12. Неизменяемый снимок входов

Контрольные суммы входов этапов 1 и 2 фиксируются до обучения. Dev qrels пока отсутствует локально и связан только с объявленной в манифестах SHA-256.


In [ ]:
snapshot_command = (
    "import json; from rusearchrank.training_data import "
    "load_finetune_config, phase12_immutable_snapshot; "
    "c=load_finetune_config('configs/finetune.yaml'); "
    "print(json.dumps(phase12_immutable_snapshot(c, require_all=True), "
    "sort_keys=True))"
)
phase12_snapshot = json.loads(
    run_checked(
        [RUN_PYTHON, "-c", snapshot_command],
        cwd=REPOSITORY,
    ).stdout
)
snapshot_path = (
    REPOSITORY / "artifacts/work/phase3/phase12_preselection_snapshot.json"
)
snapshot_path.parent.mkdir(parents=True, exist_ok=True)
serialized_snapshot = json.dumps(phase12_snapshot, sort_keys=True) + "\n"
if snapshot_path.is_file():
    if snapshot_path.read_text(encoding="utf-8") != serialized_snapshot:
        raise RuntimeError("Снимок входов изменился между повторными запусками")
else:
    snapshot_path.write_text(serialized_snapshot, encoding="utf-8")
if phase12_snapshot.get(train_qrels_relative) != TRAIN_QRELS_SHA256:
    raise RuntimeError("Снимок не содержит ожидаемую SHA-256 train qrels")
print({"input_count": len(phase12_snapshot), "train_qrels_sha256": TRAIN_QRELS_SHA256})


## Этап 13. Проверка исходной модели

Zero-shot контрольная точка оценивается только на внутренней контрольной выборке из train.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "validate-checkpoint",
        "--config",
        CONFIG,
        "--checkpoint",
        "base",
    ],
    cwd=REPOSITORY,
    stream=True,
)
validation_metrics = json.loads(
    (
        REPOSITORY / "reports/metrics/validation_checkpoint_metrics.json"
    ).read_text(encoding="utf-8")
)
print(
    {
        "candidate": "S0",
        "validation_ndcg_at_10": validation_metrics["S0"]["ndcg_at_10"],
    }
)


## Этап 14. Полная первичная проверка

Smoke выполняет четыре настоящих шага оптимизатора. Пропускная способность считается по трём полным шагам после разогрева; создание контрольной точки и ZIP измеряется отдельно.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "smoke-finetune",
        "--config",
        CONFIG,
        "--limit-pairs",
        "64",
    ],
    cwd=REPOSITORY,
    stream=True,
)
smoke = json.loads(
    (REPOSITORY / "reports/audit/finetune_smoke.json").read_text(
        encoding="utf-8"
    )
)
required_smoke_values = {
    "status": "PASS",
    "real_model_forward": True,
    "real_optimizer_step": True,
    "fixture_only": False,
    "device": "cuda",
    "checkpoint_save_load_roundtrip": True,
    "resume_state_roundtrip": True,
    "zip_hash_roundtrip": True,
}
smoke_mismatches = {
    key: {"actual": smoke.get(key), "expected": expected}
    for key, expected in required_smoke_values.items()
    if smoke.get(key) != expected
}
if smoke_mismatches:
    raise RuntimeError(f"Полная первичная проверка не пройдена: {smoke_mismatches}")
if smoke.get("dtype") != "float32":
    raise RuntimeError("Smoke должен выполняться в float32")
resource = json.loads(
    (REPOSITORY / "reports/audit/resource_report.json").read_text(
        encoding="utf-8"
    )
)
if resource.get("status") != "PASS":
    raise RuntimeError(
        "Оценка ресурсов ненадёжна: "
        f"{resource.get('status')} — {resource.get('reason')}"
    )
FINETUNE_SMOKE_PASSED = True
print(
    {
        "smoke_status": smoke["status"],
        "steady_state_pairs_per_second": resource[
            "steady_state_pairs_per_second"
        ],
        "estimated_training_time_range_seconds": resource[
            "estimated_training_time_range_seconds"
        ],
    }
)


## Этап 15. Контроль C1

C1 проверяет путь обучения на перемешанных метках. FAIL и BLOCKED_FOR_REVIEW останавливают протокол.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "finetune",
        "--config",
        CONFIG,
        "--run-id",
        "C1",
    ]
    + training_options("C1"),
    cwd=REPOSITORY,
    stream=True,
)
control = json.loads(
    (REPOSITORY / "reports/audit/control_c1.json").read_text(encoding="utf-8")
)
if control.get("status") in {"FAIL", "BLOCKED_FOR_REVIEW"}:
    raise RuntimeError(
        f"C1 остановил протокол: {control.get('status')} — {control.get('reason')}"
    )
print({"C1": control.get("status"), "mean_delta": control.get("mean_delta")})


## Этап 16. Запуск A1

A1 обучается только на парах с экспертно оценёнными отрицательными примерами при скорости обучения 7e-6.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "finetune",
        "--config",
        CONFIG,
        "--run-id",
        "A1",
    ]
    + training_options("A1"),
    cwd=REPOSITORY,
    stream=True,
)


## Этап 17. Запуск A2

A2 использует тот же режим данных и скорость обучения 2e-5.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "finetune",
        "--config",
        CONFIG,
        "--run-id",
        "A2",
    ]
    + training_options("A2"),
    cwd=REPOSITORY,
    stream=True,
)


## Этап 18. Запуск B1

B1 использует weak negatives — документы без экспертной оценки, выбранные как слабые отрицательные примеры. Скорость обучения наследуется от лучшего из A1/A2 по внутренней контрольной выборке.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "finetune",
        "--config",
        CONFIG,
        "--run-id",
        "B1",
    ]
    + training_options("B1"),
    cwd=REPOSITORY,
    stream=True,
)


## Этап 19. Единственный выбор контрольной точки

Решение публикуется атомарно до доступа к dev. Повторный запуск команды разрешён только как переиспользование побайтно идентичного решения.


In [ ]:
ledger_path = REPOSITORY / "reports/audit/dev_access_ledger.jsonl"
selection_path = REPOSITORY / "reports/audit/checkpoint_selection.json"
if ledger_path.is_file() and ledger_path.stat().st_size and not selection_path.is_file():
    raise RuntimeError("До выбора уже существует непустой журнал доступа к dev")
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "select-checkpoint",
        "--config",
        CONFIG,
    ],
    cwd=REPOSITORY,
    stream=True,
)
selection = json.loads(
    (
        REPOSITORY / "reports/audit/checkpoint_selection.json"
    ).read_text(encoding="utf-8")
)
selection_body = {
    key: value for key, value in selection.items() if key != "selection_sha256"
}
computed_selection_sha256 = hashlib.sha256(
    json.dumps(
        selection_body,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
if selection.get("selection_sha256") != computed_selection_sha256:
    raise RuntimeError("Контрольная сумма решения о выборе не совпала")
if selection.get("selection_written_before_dev_access") is not True:
    raise RuntimeError("Не подтверждён порядок выбора до доступа к dev")
print(
    {
        "selected_run_id": selection["best_finetuned_checkpoint"]["run_id"],
        "selected_epoch": selection["best_finetuned_checkpoint"]["epoch"],
        "production_kind": selection["production_system"]["kind"],
        "selection_sha256": selection["selection_sha256"],
    }
)


## Этап 20. Первый разрешённый доступ к dev qrels

Команда сначала добавляет связанное с выбором событие в цепочку контрольных сумм журнала, затем получает и проверяет dev qrels.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "prepare-dev-evaluation",
        "--config",
        CONFIG,
    ],
    cwd=REPOSITORY,
    stream=True,
)
prepared_qrels = json.loads(
    (
        REPOSITORY / "artifacts/work/phase3/prepared_dev_qrels.json"
    ).read_text(encoding="utf-8")
)
print(
    {
        "row_count": prepared_qrels["row_count"],
        "query_count": prepared_qrels["query_count"],
        "source_sha256": prepared_qrels["source_sha256"],
    }
)


## Этап 21. Расчёт оценок fine-tuned модели

Выбранная модель рассчитывает оценки ровно для зафиксированного top-100; состав кандидатов не меняется.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "score-finetuned",
        "--config",
        CONFIG,
    ]
    + phase3_options(),
    cwd=REPOSITORY,
    stream=True,
)


## Этап 22. Итоговое оценивание

BM25, zero-shot и fine-tuned системы сравниваются на одном множестве кандидатов. В интерфейс выводятся только сводные метрики; подробности остаются в JSON.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "evaluate-phase3",
        "--config",
        CONFIG,
    ]
    + phase3_options(),
    cwd=REPOSITORY,
    stream=True,
)
comparison = json.loads(
    (
        REPOSITORY / "reports/metrics/dev_three_way_comparison.json"
    ).read_text(encoding="utf-8")
)
print(
    {
        "pipeline_status": comparison["pipeline_status"],
        "systems": {
            name: {
                "ndcg_at_10": values["ndcg_at_10"],
                "mrr_at_10": values["mrr_at_10"],
                "recall_at_100": values["recall_at_100"],
            }
            for name, values in comparison["systems"].items()
        },
        "primary_delta": comparison[
            "primary_finetuned_minus_zero_shot"
        ]["mean_delta"],
        "ml_outcome": comparison["ml_outcome"]["label"],
    }
)


## Этап 23. Упаковка результатов

Создаются новые архивы выпуска v1.0.1. Старые официальные ZIP не открываются на запись и не перезаписываются.


In [ ]:
run_checked(
    [
        RUN_PYTHON,
        "-m",
        "rusearchrank.cli",
        "package-phase3",
        "--config",
        CONFIG,
    ]
    + phase3_options(),
    cwd=REPOSITORY,
    stream=True,
)


## Этап 24. CRC/SHA-проверка и публикация ZIP

Оба архива повторно проверяются, после чего копируются в отдельный каталог внутри рабочего пространства Kaggle. Существующий файл с другими байтами не заменяется.


In [ ]:
FINAL_OUTPUT = KAGGLE_WORKING / "phase3-final"
FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)
result_zip = REPOSITORY / config["archive"]["results_zip"]
selected_run_id = selection["best_finetuned_checkpoint"]["run_id"]
model_zip = REPOSITORY / config["archive"]["model_zip_template"].format(
    run_id=selected_run_id
)
final_reports = []
for source in (result_zip, model_zip):
    if not source.is_file():
        raise RuntimeError(f"Не найден итоговый архив: {source}")
    with zipfile.ZipFile(source) as archive:
        names = archive.namelist()
        for name in names:
            member = PurePosixPath(name)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Небезопасный путь в итоговом ZIP: {name}")
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"CRC-проверка не пройдена: {bad_member}")
        if source == result_zip:
            manifest = json.loads(
                archive.read(
                    "reports/audit/training_manifest.json"
                ).decode("utf-8")
            )
            for entry in manifest["entries"]:
                payload = archive.read(entry["path"])
                actual = hashlib.sha256(payload).hexdigest()
                if actual != entry["file_sha256"]:
                    raise RuntimeError(
                        f"Манифест не совпал с ZIP: {entry['path']}"
                    )
    source_sha256 = sha256_file(source)
    destination = FINAL_OUTPUT / source.name
    if destination.exists():
        if not destination.is_file() or sha256_file(destination) != source_sha256:
            raise RuntimeError(
                f"Запрещена замена другого итогового архива: {destination}"
            )
    else:
        shutil.copyfile(source, destination)
    if sha256_file(destination) != source_sha256:
        raise RuntimeError(f"Ошибка копирования итогового архива: {destination}")
    final_reports.append(
        {
            "path": str(destination),
            "size_bytes": destination.stat().st_size,
            "sha256": source_sha256,
        }
    )
print(json.dumps(final_reports, ensure_ascii=False, indent=2))
print("Архивы готовы в рабочем каталоге Kaggle; GPU-сессию можно завершить.")
